In [1]:
!pip -q install langchain langchain-community langchain-core sentence-transformers faiss-cpu groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 46.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 44.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 5.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [2]:
from google.colab import userdata

try:
    GROQ_API_KEY = userdata.get("GROQ_API_KEY")

    if not GROQ_API_KEY:
        raise ValueError("GROQ_API_KEY not found.")

    print("✅ API Key Loaded Successfully.")

except Exception as e:
    print("❌", e)

✅ API Key Loaded Successfully.


In [5]:
from groq import Groq

from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

/tmp/ipykernel_811/2602054899.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.embeddings import HuggingFaceEmbeddings


In [6]:
sample_text = """
Climate change refers to long-term changes in Earth's temperature and weather patterns.

Human activities such as burning coal, oil and natural gas release greenhouse gases.

These gases trap heat in the atmosphere.

Climate change causes rising sea levels, melting glaciers, stronger storms,
heatwaves, floods, droughts and damage to ecosystems.

Countries are trying to reduce emissions by using renewable energy,
protecting forests and improving energy efficiency.
"""

documents = [Document(page_content=sample_text)]

In [7]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=30
)

chunks = splitter.split_documents(documents)

print("Chunks Created:", len(chunks))

Chunks Created: 2


In [8]:
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

vectorstore = FAISS.from_documents(
    chunks,
    embeddings
)

print("✅ FAISS Index Created")

/tmp/ipykernel_811/1414926504.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ FAISS Index Created


In [9]:
retriever = vectorstore.as_retriever(
    search_kwargs={"k":2}
)

In [10]:
client = Groq(api_key=GROQ_API_KEY)

In [11]:
def rag(question):

    docs = retriever.invoke(question)

    context = "\n\n".join(
        doc.page_content for doc in docs
    )

    prompt = f"""
You are a helpful assistant.

Use ONLY the context below.

If the answer is not present,
reply only:

I don't know.

Context:
{context}

Question:
{question}

Answer:
"""

    response = client.chat.completions.create(

        model="llama-3.1-8b-instant",

        temperature=0,

        messages=[
            {
                "role":"user",
                "content":prompt
            }
        ]
    )

    answer = response.choices[0].message.content

    return answer, docs

In [12]:
questions = [

"What causes climate change?",

"What are the effects of climate change?"

]

for q in questions:

    answer, docs = rag(q)

    print("="*80)
    print("Question:")
    print(q)

    print("\nAnswer:")
    print(answer)

    print("\nRetrieved Chunks:\n")

    for d in docs:
        print("-", d.page_content)
        print()

Question:
What causes climate change?

Answer:
Human activities such as burning coal, oil and natural gas release greenhouse gases. These gases trap heat in the atmosphere.

Retrieved Chunks:

- Climate change causes rising sea levels, melting glaciers, stronger storms,
heatwaves, floods, droughts and damage to ecosystems.

Countries are trying to reduce emissions by using renewable energy,
protecting forests and improving energy efficiency.

- Climate change refers to long-term changes in Earth's temperature and weather patterns.

Human activities such as burning coal, oil and natural gas release greenhouse gases.

These gases trap heat in the atmosphere.

Question:
What are the effects of climate change?

Answer:
Rising sea levels, melting glaciers, stronger storms, heatwaves, floods, droughts, and damage to ecosystems.

Retrieved Chunks:

- Climate change causes rising sea levels, melting glaciers, stronger storms,
heatwaves, floods, droughts and damage to ecosystems.

Countries are

In [13]:
print("="*80)
print("ANALYSIS")
print("="*80)

print("""
1. The FAISS retriever searched the indexed document.

2. The top two relevant chunks were retrieved.

3. The retrieved chunks were inserted into the prompt.

4. Groq Llama generated the answer using only the retrieved context.

5. This demonstrates a basic Retrieval-Augmented Generation (RAG)
pipeline using LangChain concepts and LCEL-style retrieval flow.
""")

ANALYSIS

1. The FAISS retriever searched the indexed document.

2. The top two relevant chunks were retrieved.

3. The retrieved chunks were inserted into the prompt.

4. Groq Llama generated the answer using only the retrieved context.

5. This demonstrates a basic Retrieval-Augmented Generation (RAG)
pipeline using LangChain concepts and LCEL-style retrieval flow.

